# Data Preprocessing

First things first, we must import the required libraries and read in the data.

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [12]:
data = pd.read_csv('C:/Users/poke5/Desktop/Projects/ML-Churn/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

After reading in the data, we must start our data preprocessing. First steps include categorizing the total charges to replace blank cells with nas. Then we must account for any nas and replace them with either mean or mode depending on whether the data is categorical/numerical. 

In [13]:
# Cateorize toal charges column
if 'TotalCharges' in data.columns:
    blank_mask = data['TotalCharges'].astype(str).str.strip() == ''
    if blank_mask.any():
        data.loc[blank_mask, 'TotalCharges'] = np.nan
    data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

num_cols = data.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = data.select_dtypes(exclude=[np.number]).columns.tolist()

# Fill nas with median values in numeric columns
for col in num_cols:            
    med = data[col].median()
    if pd.isna(med):
        print(f'Column {col} is all NaN; no median available')
    else:
        data[col].fillna(med, inplace=True)

# Fill nas with mode in categorical columns
for col in cat_cols:
    try:
        mode_val = data[col].mode(dropna=True)
        if not mode_val.empty:
            data[col].fillna(mode_val[0], inplace=True)
    except Exception as e:
        print(f'Could not compute mode for {col}: {e}')

data.head()

C:\Users\poke5\AppData\Local\Temp\ipykernel_26912\826048311.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(med, inplace=True)
C:\Users\poke5\AppData\Local\Temp\ipykernel_26912\826048311.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when do

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Feature Engineering: Customer Metrics
I have derived two other variables to help better understand customer patterns/trends. 
1. **Num_Services**: Counts the total number of services each customer has subscribed to
   - Includes the following services:
     * Phone Service
     * Internet Service
     * Online Security
     * Online Backup
     * Device Protection
     * Tech Support
     * Streaming TV
     * Streaming Movies

2. **is_high_value**: Binary value indicating high-spending customers
   - Identifies customers with monthly charges above the 75th percentile
   - Useful for analyzing churn patterns among high-value customers
   - Formula: 1 if MonthlyCharges > 75th percentile, 0 otherwise

In [14]:
# Calculate number of services
service_columns = [
    'PhoneService', 
    'InternetService',  
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]

data['Num_Services'] = 0

# Count services (yes or No)
for service in service_columns:
    if service in ['PhoneService', 'InternetService']:
        data['Num_Services'] += (data[service] == 'Yes').astype(int)
    else:
        data['Num_Services'] += ((data[service] != 'No') & (data[service] != 'No internet service')).astype(int)

# Calculate high-value customers
percentile_75 = data['MonthlyCharges'].quantile(0.75)
data['is_high_value'] = (data['MonthlyCharges'] > percentile_75).astype(int)

In [15]:
display(data[['tenure', 'MonthlyCharges', 'TotalCharges', 'Num_Services', 'is_high_value']].head(10))

,tenure,MonthlyCharges,TotalCharges,Num_Services,is_high_value
0,1,29.85,29.85,1,0
1,34,56.95,1889.50,3,0
2,2,53.85,108.15,3,0
3,45,42.30,1840.75,3,0
4,2,70.70,151.65,1,0
5,8,99.65,820.50,4,1
6,22,89.10,1949.40,3,0
7,10,29.75,301.90,1,0
8,28,104.80,3046.05,5,1
9,62,56.15,3487.95,3,0


# Data Splitting

Now we'll split our preprocessed dataset into three parts:
- Training set (70%): Used to train our models
- Test set (20%): Used for final model evaluation
- Validation set (10%): Used for model selection and hyperparameter tuning

In [16]:
# Separate target from features
X = data.drop('Churn', axis=1)
y = data['Churn']

# First split: 70% train, 30% remaining
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,   
    random_state=42,  
    stratify=y        
)

# Second split: Split remaining 30% into 20% test and 10% validation
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.33,  
    random_state=42,
    stratify=y_temp
)

# Save Processed Datasets

After splitting the data, I will write the three datasets (train, test, validation) out to csv files. This will allow for them to be read into a new notebook or script, ensuring reproducibility. 

In [17]:
# Create processed_data directory if it doesn't exist

train_data = X_train.copy()
train_data['Churn'] = y_train
test_data = X_test.copy()
test_data['Churn'] = y_test
val_data = X_val.copy()
val_data['Churn'] = y_val

# Save datasets to CSV files
train_data.to_csv(f'{processed_data_dir}/train_data.csv', index=False)
test_data.to_csv(f'{processed_data_dir}/test_data.csv', index=False)
val_data.to_csv(f'{processed_data_dir}/validation_data.csv', index=False)

print("Datasets saved to processed_data directory.")

Datasets saved to processed_data directory.
